# staging

In [1]:
# dependencies

#pip install xlsxwriter


In [2]:
# imports and settings

from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.utils import get_column_letter
from openpyxl import Workbook, load_workbook
from typing import List, Tuple, Dict
from zipfile import BadZipFile
from itertools import product

import pandas as pd
import numpy as np
import difflib
import re
import os

# pandas optional preferences
pd.set_option("display.float_format", "{:,.6f}".format)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)


In [3]:
# directory & output staging
"""
python file exists one level above "raw_data" folder
"""

DATA_DIR = "_raw_data"
OUT_DIR = "_clean_data"
os.makedirs(OUT_DIR, exist_ok=True)

AUDIT_XLSX = os.path.join(OUT_DIR, "_clean_audit.xlsx")

if not os.path.exists(AUDIT_XLSX):
    wb = Workbook()
    wb.save(AUDIT_XLSX)
else:
    # load if valid; recreate if corrupted
    try:
        wb = load_workbook(AUDIT_XLSX)
    except BadZipFile:
        wb = Workbook()
        wb.save(AUDIT_XLSX)
        wb = load_workbook(AUDIT_XLSX)

    # detect whether workbook contains any cell values
    has_data = False
    for ws in wb.worksheets:
        for row in ws.iter_rows(values_only=True):
            if any(v is not None for v in row):
                has_data = True
                break
        if has_data:
            break

    if has_data:
        for name in list(wb.sheetnames):
            wb.remove(wb[name])
        wb.create_sheet("Sheet")
        wb.active = wb["Sheet"]
        wb.save(AUDIT_XLSX)


In [4]:
# Guardrail for XLSX/PDF inputs (non-intrusive)
# Purpose:
# - Detect XLSX/PDF files in DATA_DIR
# - For XLSX: attempt a conservative "rectangularity" check; only ingest if it looks like a single clean table
# - For PDF: never ingest; always flag manual_required
# - Log all non-CSV handling decisions to an ingestion_exceptions table you can write into auto_audit

def list_input_files(data_dir: str) -> dict[str, list[str]]:
    """
    note: Lists input files by extension in a directory. Returns relative filenames (not full paths).
    """
    out = {"csv": [], "xlsx": [], "xls": [], "pdf": [], "other": []}
    for f in os.listdir(data_dir):
        if not os.path.isfile(os.path.join(data_dir, f)):
            continue
        ext = os.path.splitext(f)[1].lower()
        if ext == ".csv":
            out["csv"].append(f)
        elif ext == ".xlsx":
            out["xlsx"].append(f)
        elif ext == ".xls":
            out["xls"].append(f)
        elif ext == ".pdf":
            out["pdf"].append(f)
        else:
            out["other"].append(f)
    return out

def _rectangularity_checks(df: pd.DataFrame) -> list[str]:
    """
    note: Conservative structure checks for a single-table sheet. Returns issue codes; empty list means "looks safe".
    """
    issues: list[str] = []
    if df is None or df.empty:
        return ["empty_sheet_or_failed_read"]

    # Many "Unnamed" columns often indicates multi-row headers or misaligned export
    col_unnamed = sum(str(c).strip().lower().startswith("unnamed") for c in df.columns)
    if len(df.columns) > 0 and (col_unnamed / len(df.columns)) >= 0.5:
        issues.append("likely_multiheader_unnamed_columns")

    # Row-wise non-null counts should be relatively stable in a rectangular table
    nonnull_counts = df.notna().sum(axis=1).astype(float)
    if len(nonnull_counts) > 0:
        mean = float(nonnull_counts.mean())
        std = float(nonnull_counts.std(ddof=0))
        if mean == 0:
            issues.append("mostly_blank_rows")
        else:
            cv = std / mean
            # High variation suggests sections/subtables/notes in the same sheet
            if cv > 0.35:
                issues.append("nonrectangular_row_density_high")

        blank_row_frac = float((nonnull_counts == 0).mean())
        if blank_row_frac > 0.20:
            issues.append("many_blank_rows")

    return issues

def inspect_xlsx_for_safe_ingest(xlsx_path: str, *, max_rows: int = 5000) -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    """
    note: Inspects an XLSX file and attempts to ingest only sheets that look like a single rectangular table.
    Returns (dfs_ingested, issues_df). Any sheet with issues is not ingested and is logged as manual_required.
    """
    dfs_ingested: dict[str, pd.DataFrame] = []
    issues_rows: list[dict] = []

    wb = load_workbook(xlsx_path, read_only=True, data_only=True)
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]

        # Merged cells are a strong indicator of semantic layout
        if ws.merged_cells and len(ws.merged_cells.ranges) > 0:
            issues_rows.append(
                {
                    "file": os.path.basename(xlsx_path),
                    "file_type": "xlsx",
                    "sheet": sheet_name,
                    "status": "manual_required",
                    "reason": "merged_cells_detected",
                    "details": f"merged_ranges={len(ws.merged_cells.ranges)}",
                }
            )
            continue

        # Try reading as a normal table
        try:
            df = pd.read_excel(xlsx_path, sheet_name=sheet_name, engine="openpyxl")
        except Exception as e:
            issues_rows.append(
                {
                    "file": os.path.basename(xlsx_path),
                    "file_type": "xlsx",
                    "sheet": sheet_name,
                    "status": "manual_required",
                    "reason": "read_excel_failed",
                    "details": str(e)[:250],
                }
            )
            continue

        # Optional row cap to avoid memory surprises
        if max_rows is not None and len(df) > int(max_rows):
            issues_rows.append(
                {
                    "file": os.path.basename(xlsx_path),
                    "file_type": "xlsx",
                    "sheet": sheet_name,
                    "status": "manual_required",
                    "reason": "too_many_rows_for_auto_ingest",
                    "details": f"rows={len(df)} cap={max_rows}",
                }
            )
            continue

        issues = _rectangularity_checks(df)
        if issues:
            issues_rows.append(
                {
                    "file": os.path.basename(xlsx_path),
                    "file_type": "xlsx",
                    "sheet": sheet_name,
                    "status": "manual_required",
                    "reason": ";".join(issues),
                    "details": "",
                }
            )
            continue

        # Safe ingest: name includes file base + sheet
        base = os.path.splitext(os.path.basename(xlsx_path))[0]
        key = f"{base}__{sheet_name}".strip()
        dfs_ingested[key] = df
        issues_rows.append(
            {
                "file": os.path.basename(xlsx_path),
                "file_type": "xlsx",
                "sheet": sheet_name,
                "status": "auto_ingested",
                "reason": "",
                "details": f"table_key={key}",
            }
        )

    issues_df = pd.DataFrame(
        issues_rows,
        columns=["file", "file_type", "sheet", "status", "reason", "details"],
    )
    return dfs_ingested, issues_df

def build_ingestion_exceptions(data_dir: str, *, ingest_safe_xlsx: bool = True) -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    """
    note: Guardrail wrapper. Detects xlsx/pdf and (optionally) ingests only safe xlsx sheets.
    Returns (dfs_extra, ingestion_exceptions_df). PDFs are always logged as manual_required.
    """
    files = list_input_files(data_dir)

    dfs_extra: dict[str, pd.DataFrame] = {}
    issues_frames: list[pd.DataFrame] = []

    # XLSX/XLS handling
    if ingest_safe_xlsx:
        for f in files["xlsx"]:
            full = os.path.join(data_dir, f)
            dfs_ingested, issues_df = inspect_xlsx_for_safe_ingest(full)
            dfs_extra.update(dfs_ingested)
            issues_frames.append(issues_df)
    else:
        for f in files["xlsx"]:
            issues_frames.append(
                pd.DataFrame(
                    [{
                        "file": f,
                        "file_type": "xlsx",
                        "sheet": "",
                        "status": "manual_required",
                        "reason": "xlsx_auto_ingest_disabled",
                        "details": "",
                    }],
                    columns=["file", "file_type", "sheet", "status", "reason", "details"],
                )
            )

    # PDFs are always manual for this pipeline
    for f in files["pdf"]:
        issues_frames.append(
            pd.DataFrame(
                [{
                    "file": f,
                    "file_type": "pdf",
                    "sheet": "",
                    "status": "manual_required",
                    "reason": "pdf_not_auto_ingested",
                    "details": "",
                }],
                columns=["file", "file_type", "sheet", "status", "reason", "details"],
            )
        )

    ingestion_exceptions = (
        pd.concat(issues_frames, ignore_index=True)
        if issues_frames
        else pd.DataFrame(columns=["file", "file_type", "sheet", "status", "reason", "details"])
    )

    return dfs_extra, ingestion_exceptions


In [5]:
# load raw inputs:
# - Loads CSVs
# - Applies the non-intrusive guardrail for XLSX/PDF
# - Optionally ingests only safe XLSX sheets into dfs_raw
# - Keeps an ingestion_exceptions table to write into auto_audit later

csv_files = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(".csv")]

dfs_raw = {
    f.replace(".csv", ""): pd.read_csv(os.path.join(DATA_DIR, f))
    for f in csv_files
}

dfs_extra, ingestion_exceptions = build_ingestion_exceptions(DATA_DIR, ingest_safe_xlsx=True)
dfs_raw.update(dfs_extra)

print(f"Loaded CSV tables: {len(csv_files)}")
print(f"Auto-ingested XLSX sheets: {len(dfs_extra)}")
print(f"Ingestion exceptions rows: {len(ingestion_exceptions)}")
print(ingestion_exceptions[["status","reason","details"]].head(5).to_string(index=False))
display(ingestion_exceptions)


Loaded CSV tables: 3
Auto-ingested XLSX sheets: 0
Ingestion exceptions rows: 0
Empty DataFrame
Columns: [status, reason, details]
Index: []


,file,file_type,sheet,status,reason,details


In [6]:
# exploration dfs

dfs = {}

for f in csv_files:
    name = f.replace(".csv", "")
    path = os.path.join(DATA_DIR, f)
    dfs[name] = pd.read_csv(path)

# quick sanity check
{key: df.shape for key, df in dfs.items()}


{'GSCM 521 - Banner Grade Data v2': (880, 22),
 'GSCM 521 - Canvas Gradebook Data': (793, 3),
 'GSCM 521 - Canvas Interaction Data v2': (324148, 15)}

In [7]:
# quick head assessments

for name, df in dfs.items():
    print("\n" + name)
    display(df.head(3))



GSCM 521 - Banner Grade Data v2


,Subject,Course,Title,Term,Random Student,Random ID,Grade,Instructor,CRN,Reg_Status,Unduplicated_Race,Citizenship,Overall_UG_Credits,Overall_GR_Credits,Credits_in_Class_Term,1st_Term_at_PSU,Latest_Term_with_Reg_Activity,Level,Latest_Class,Major_1_in_Class_Term,Latest_Major_1,Random Email
0,ACTG,335,Ais And Accounting Analytics,202204,"Aem, Udf",840806293,A,"Kaufman, Matt",10003,RW,White,Citizen,279.500000,NaN,10.000000,201504,202302,NaN,Non-Admit,PBAC--Postbac Account Cert,0000--Undeclared/Not Applicable,UdfAem@pdx.edu
1,ACTG,335,Ais And Accounting Analytics,202204,"Anp, Spo",821380792,C,"Kaufman, Matt",10002,RW,White,Citizen,196.000000,NaN,16.000000,202104,202304,NaN,Non-Admit,ACTG--Business Adm: Accounting,0000--Undeclared/Not Applicable,SpoAnp@pdx.edu
2,ACTG,335,Ais And Accounting Analytics,202204,"Bcq, Zqx",803055564,B-,"Kaufman, Matt",10003,RW,Black or African American,Citizen,150.000000,NaN,12.000000,201904,202302,UG,Senior,ACTG--Business Adm: Accounting,MAL--Bus Adm: Mgmt & Leadership,ZqxBcq@pdx.edu



GSCM 521 - Canvas Gradebook Data


,Random Student,Random SIS User ID,Random SIS Login ID
0,"Tse, Sml",iuybybxh-7b98-4c40-915a-bhuazfvgugyp,SmlTse
1,"Iwu, Xts",hmtdikpg-862d-4948-a099-ilvohytgsfdl,XtsIwu
2,"Iwu, Xts",hmtdikpg-862d-4948-a099-ilvohytgsfdl,XtsIwu



GSCM 521 - Canvas Interaction Data v2


,Student Id,Random Student Name,Random Sortable Name,Random Student SIS ID,Section Id,Section Name,Course Id,Course Name,Content Type,Content Name,Times Viewed,Times Participated,Start Date,First Viewed,Last Viewed
0,"200,830,000,000,000,000.000000",Qcx Pyt,"Pyt, Qcx",izbtazwy-1fdc-4d45-98a6-mxwybumlbjgm,"200,830,000,000,000,000.000000",ACTG-335-HB1: Ais And Accounting Analytics,"200,830,000,000,000,000.000000",Ais And Accounting Analytics HB1 Fall 2022,course.files.file,ACTG 335_F22_Kaufman_Syllabus-1.doc,3,0,9/19/2022,9/19/2022 16:29,9/19/2022 17:04
1,"200,830,000,000,000,000.000000",Qcx Pyt,"Pyt, Qcx",izbtazwy-1fdc-4d45-98a6-mxwybumlbjgm,"200,830,000,000,000,000.000000",ACTG-335-HB1: Ais And Accounting Analytics,"200,830,000,000,000,000.000000",Ais And Accounting Analytics HB1 Fall 2022,course.home,Course Home,4,0,9/19/2022,9/19/2022 16:29,9/19/2022 17:17
2,"200,830,000,000,000,000.000000",Qcx Pyt,"Pyt, Qcx",izbtazwy-1fdc-4d45-98a6-mxwybumlbjgm,"200,830,000,000,000,000.000000",ACTG-335-HB1: Ais And Accounting Analytics,"200,830,000,000,000,000.000000",Ais And Accounting Analytics HB1 Fall 2022,course.syllabus,Course Syllabus,3,0,9/19/2022,9/19/2022 16:29,9/19/2022 16:52


# core utilities

In [8]:
# col norm/sampling/mapping and delimiter funcs

def normalize_col_name_simple(name: str) -> str:
    """
    note: Minimal normalization only: strip leading/trailing spaces, lowercase,
    replace internal whitespace runs with underscores. Does not remove punctuation.
    """
    s = str(name).strip().lower()
    s = re.sub(r"\s+", "_", s)
    return s

def make_unique_names(names: list[str]) -> tuple[list[str], pd.DataFrame]:
    """
    note: Forces uniqueness of column names by suffixing _2, _3, ... on collisions.
    Returns (unique_names, collisions_log_df) where collisions_log_df is empty when none.
    """
    seen: dict[str, int] = {}
    out: list[str] = []
    rows: list[dict] = []

    for n in names:
        if n not in seen:
            seen[n] = 1
            out.append(n)
        else:
            seen[n] += 1
            new = f"{n}_{seen[n]}"
            out.append(new)
            rows.append({"original_normalized": n, "resolved_name": new, "collision_n": seen[n]})

    return out, pd.DataFrame(rows)

def build_column_map(table_name: str, df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    note: Builds a reversible column mapping using only minimal normalization.
    No table prefixing. Collision resolution is applied only if needed.
    """
    raw_cols = list(df.columns)
    norm_cols = [normalize_col_name_simple(c) for c in raw_cols]

    final_cols, collisions = make_unique_names(norm_cols)

    col_map = pd.DataFrame(
        {
            "table": table_name,
            "raw_column": raw_cols,
            "normalized_column": norm_cols,
            "final_column": final_cols,
        }
    )
    return col_map, collisions

def apply_column_map(df: pd.DataFrame, col_map: pd.DataFrame) -> pd.DataFrame:
    """
    note: Applies a column mapping (raw_column -> final_column) to a DataFrame.
    Returns a copy and does not mutate the input DataFrame.
    """
    rename_dict = dict(zip(col_map["raw_column"], col_map["final_column"]))
    return df.rename(columns=rename_dict).copy()

def drop_pandas_index_artifacts(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    note: Drops common pandas CSV index artifacts like 'Unnamed: 0' (case-insensitive).
    Returns (clean_df, dropped_cols_df) for audit traceability.
    """
    cols = list(df.columns)
    drop_cols = [c for c in cols if str(c).strip().lower().startswith("unnamed")]
    dropped = pd.DataFrame({"dropped_column": drop_cols})
    return df.drop(columns=drop_cols, errors="ignore").copy(), dropped

def detect_delimited_columns(
    df: pd.DataFrame,
    *,
    delimiters: tuple[str, ...] = (",", ";", "|", "/", "\\"),
    min_nonnull: int = 200,
    sample_n: int = 2000,
    min_delim_rate: float = 0.25,
    min_mean_delims_per_hit: float = 1.2,
    max_unique_ratio: float = 0.95,
) -> pd.DataFrame:
    """
    note: Detects likely 'multi-attribute-in-one-cell' columns by scanning object columns for delimiter patterns.
    Returns a table of candidates with rates and a recommended delimiter.
    """
    n = len(df)
    if n == 0:
        return pd.DataFrame(columns=["column","dtype","nonnull","unique_ratio","delimiter","delim_hit_rate","mean_delims_when_hit"])

    rows: list[dict] = []

    for col in df.columns:
        s = df[col]
        if not pd.api.types.is_object_dtype(s) and not pd.api.types.is_string_dtype(s):
            continue

        nonnull = int(s.notna().sum())
        if nonnull < int(min_nonnull):
            continue

        # sample to keep it fast and stable
        ss = s.dropna().astype(str)
        if len(ss) > int(sample_n):
            ss = ss.sample(n=int(sample_n), random_state=0)

        # exclude columns that are essentially unique IDs (high unique_ratio)
        nun = int(s.nunique(dropna=True))
        unique_ratio = (nun / nonnull) if nonnull else np.nan
        if pd.notna(unique_ratio) and unique_ratio > float(max_unique_ratio):
            continue

        best = None
        for d in delimiters:
            counts = ss.str.count(re.escape(d))
            hits = counts[counts > 0]
            hit_rate = float((counts > 0).mean()) if len(counts) else 0.0
            mean_when_hit = float(hits.mean()) if len(hits) else 0.0

            if hit_rate >= float(min_delim_rate) and mean_when_hit >= float(min_mean_delims_per_hit):
                score = hit_rate * mean_when_hit
                if best is None or score > best["score"]:
                    best = {
                        "delimiter": d,
                        "delim_hit_rate": hit_rate,
                        "mean_delims_when_hit": mean_when_hit,
                        "score": score,
                    }

        if best is not None:
            rows.append(
                {
                    "column": col,
                    "dtype": str(s.dtype),
                    "nonnull": nonnull,
                    "unique_ratio": float(unique_ratio) if pd.notna(unique_ratio) else np.nan,
                    "delimiter": best["delimiter"],
                    "delim_hit_rate": best["delim_hit_rate"],
                    "mean_delims_when_hit": best["mean_delims_when_hit"],
                }
            )

    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["delim_hit_rate", "mean_delims_when_hit"], ascending=[False, False]).reset_index(drop=True)
    return out

def sample_delimited_values(
    df: pd.DataFrame,
    col: str,
    delimiter: str,
    *,
    n: int = 10,
) -> pd.DataFrame:
    """
    note: Provides sample raw values containing the delimiter for human review.
    """
    if col not in df.columns:
        return pd.DataFrame(columns=["column","delimiter","value"])
    s = df[col].dropna().astype(str)
    s = s[s.str.contains(re.escape(delimiter))]
    vals = s.head(int(n)).tolist()
    return pd.DataFrame([{"column": col, "delimiter": delimiter, "value": v} for v in vals])

def split_delimited_column(
    df: pd.DataFrame,
    col: str,
    delimiter: str,
    *,
    max_splits: int = 4,
    keep_original: bool = True,
    strip_tokens: bool = True,
) -> pd.DataFrame:
    """
    note: Rudimentary split of a delimited text column into multiple columns.
    Uses pandas str.split with fixed max_splits. Intended for explicit, reviewed use only.
    """
    if col not in df.columns:
        return df.copy()

    out = df.copy()
    s = out[col].astype("string")
    parts = s.str.split(delimiter, n=int(max_splits), expand=True)

    # normalize tokens
    if strip_tokens:
        for j in range(parts.shape[1]):
            parts[j] = parts[j].astype("string").str.strip()

    # name new columns
    new_cols = [f"{col}__{i+1}" for i in range(parts.shape[1])]
    for j, nc in enumerate(new_cols):
        out[nc] = parts[j]

    if not keep_original:
        out = out.drop(columns=[col])

    return out

def _empty_delimited_candidates_df() -> pd.DataFrame:
    """
    note: Provides a stable empty schema for delimited_candidates when none are detected.
    """
    return pd.DataFrame(
        columns=[
            "table","column","dtype","nonnull","unique_ratio",
            "delimiter","delim_hit_rate","mean_delims_when_hit"
        ]
    )

def _empty_delimited_samples_df() -> pd.DataFrame:
    """
    note: Provides a stable empty schema for delimited_samples when none are detected.
    """
    return pd.DataFrame(columns=["table","column","delimiter","value"])

def build_split_allowlist_template(
    delimited_candidates_all: pd.DataFrame,
    *,
    default_max_splits: int = 4,
    default_keep_original: bool = True,
    default_strip_tokens: bool = True,
) -> pd.DataFrame:
    """
    note: Builds a ready-to-copy allowlist template from detected delimiter candidates.
    This does not apply any changes; it only helps the user compose split_allowlist entries.
    """
    if delimited_candidates_all is None or delimited_candidates_all.empty:
        return pd.DataFrame(
            [{
                "table": "",
                "column": "",
                "delimiter": "",
                "max_splits": default_max_splits,
                "keep_original": default_keep_original,
                "strip_tokens": default_strip_tokens,
                "status": "none_detected",
                "notes": "No delimited candidates detected under current thresholds.",
            }],
            columns=["table","column","delimiter","max_splits","keep_original","strip_tokens","status","notes"],
        )

    tpl = delimited_candidates_all[["table","column","delimiter"]].copy()
    tpl["max_splits"] = int(default_max_splits)
    tpl["keep_original"] = bool(default_keep_original)
    tpl["strip_tokens"] = bool(default_strip_tokens)
    tpl["status"] = "proposed"
    tpl["notes"] = ""
    return tpl


In [9]:
# profiling and duplicate signatures

def profile_table(table_name: str, df: pd.DataFrame) -> pd.DataFrame:
    """
    note: Generates generic per-column profile metrics for audit and dedup readiness:
    dtype, row counts, non-null counts, unique counts, null_rate, unique_ratio
    """
    n = len(df)
    rows = []
    for c in df.columns:
        s = df[c]
        nn = int(s.notna().sum())
        nun = int(s.nunique(dropna=True))
        null_rate = 1.0 - (nn / n) if n else np.nan
        unique_ratio = (nun / nn) if nn else np.nan
        rows.append(
            {
                "table": table_name,
                "column": c,
                "dtype": str(s.dtype),
                "n_rows": int(n),
                "n_nonnull": nn,
                "n_unique": nun,
                "null_rate": float(null_rate) if pd.notna(null_rate) else np.nan,
                "unique_ratio": float(unique_ratio) if pd.notna(unique_ratio) else np.nan,
            }
        )
    return pd.DataFrame(rows)

def candidate_identity_keys(
    profile_df: pd.DataFrame,
    min_unique_ratio: float = 0.98,
    max_null_rate: float = 0.05,
    min_nonnull: int = 100,
) -> pd.DataFrame:
    """
    note: Selects candidate identity keys using only generic thresholds.
    No naming rules. Intended to propose keys for human review, not assert truth
    """
    p = profile_df.copy()
    p = p[p["n_nonnull"] >= int(min_nonnull)]
    return p[(p["unique_ratio"] >= float(min_unique_ratio)) & (p["null_rate"] <= float(max_null_rate))].sort_values(
        ["unique_ratio", "null_rate"], ascending=[False, True]
    )

def duplicate_signature(table_name: str, df: pd.DataFrame, key_cols: List[str]) -> pd.DataFrame:
    """
    note: Computes duplication signatures per candidate key column:
    dup_keys = number of distinct key values occurring >1
    dup_rows = total rows participating in duplicates (sum of counts where count>1)
    """
    rows = []
    for k in key_cols:
        s = df[k]
        nn = int(s.notna().sum())
        if nn == 0:
            rows.append({"table": table_name, "key_col": k, "dup_keys": np.nan, "dup_rows": np.nan})
            continue
        vc = s.value_counts(dropna=True)
        dup_keys = int((vc > 1).sum())
        dup_rows = int(vc[vc > 1].sum())
        rows.append({"table": table_name, "key_col": k, "dup_keys": dup_keys, "dup_rows": dup_rows})
    return pd.DataFrame(rows)

def table_shapes(dfs: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    note: Builds a compact table of shapes (rows, cols) per table for overview auditing
    """
    rows = []
    for t, df in dfs.items():
        rows.append({"table": t, "n_rows": int(df.shape[0]), "n_cols": int(df.shape[1])})
    return pd.DataFrame(rows).sort_values(["n_rows", "n_cols"], ascending=[False, False]).reset_index(drop=True)

def rename_summary(col_map_all: pd.DataFrame, collisions_all: pd.DataFrame) -> pd.DataFrame:
    """
    note: Summarizes renaming outcomes per table: number of columns, collisions, and distinct aliases
    """
    if col_map_all is None or col_map_all.empty:
        return pd.DataFrame(columns=["table", "n_cols", "table_alias", "n_collisions"])
    base = (
        col_map_all.groupby(["table", "table_alias"], as_index=False)
        .agg(n_cols=("raw_column", "count"))
        .sort_values(["n_cols"], ascending=False)
        .reset_index(drop=True)
    )
    if collisions_all is None or collisions_all.empty:
        base["n_collisions"] = 0
        return base
    c = collisions_all.groupby(["table"], as_index=False).agg(n_collisions=("collision_n", "count"))
    out = base.merge(c, on="table", how="left")
    out["n_collisions"] = out["n_collisions"].fillna(0).astype(int)
    return out

def exact_row_duplicate_summary(
    table: str,
    df: pd.DataFrame,
    *,
    sample_groups: int = 10,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    note: Detects exact duplicate rows using a stable row hash.
    Returns (summary_df, sample_groups_df). sample_groups_df shows a few hash groups and counts.
    """
    if df is None or df.empty:
        summary = pd.DataFrame([{
            "table": table,
            "n_rows": 0,
            "dup_rows": 0,
            "dup_groups": 0,
            "dup_row_rate": np.nan,
        }])
        sample = pd.DataFrame(columns=["table","row_hash","count"])
        return summary, sample

    # stable per-row hash (no normalization, treats NaN consistently)
    row_hash = pd.util.hash_pandas_object(df, index=False)
    vc = row_hash.value_counts()

    dup_groups = int((vc > 1).sum())
    dup_rows = int(vc[vc > 1].sum())
    n_rows = int(df.shape[0])
    dup_row_rate = (dup_rows / n_rows) if n_rows else np.nan

    summary = pd.DataFrame([{
        "table": table,
        "n_rows": n_rows,
        "dup_rows": dup_rows,
        "dup_groups": dup_groups,
        "dup_row_rate": float(dup_row_rate) if pd.notna(dup_row_rate) else np.nan,
    }])

    sample = vc[vc > 1].head(int(sample_groups)).reset_index()
    sample.columns = ["row_hash", "count"]
    sample.insert(0, "table", table)

    return summary, sample


def key_duplicate_offenders(
    table: str,
    df: pd.DataFrame,
    key_cols: list[str],
    *,
    top_n: int = 20,
) -> pd.DataFrame:
    """
    note: For a given key column set, returns the top offending key values (counts > 1).
    Intended for audit review only.
    """
    if df is None or df.empty:
        return pd.DataFrame(columns=["table","key_cols","key_value","count"])
    if not key_cols:
        return pd.DataFrame(columns=["table","key_cols","key_value","count"])
    missing = [c for c in key_cols if c not in df.columns]
    if missing:
        return pd.DataFrame([{
            "table": table,
            "key_cols": ",".join(key_cols),
            "key_value": "",
            "count": np.nan,
            "status": f"skipped_missing_cols:{','.join(missing)}",
        }], columns=["table","key_cols","key_value","count","status"])

    vc = df.groupby(key_cols, dropna=False).size().sort_values(ascending=False)
    vc = vc[vc > 1].head(int(top_n))

    if vc.empty:
        return pd.DataFrame(columns=["table","key_cols","key_value","count"])

    out = vc.reset_index()
    out["key_cols"] = ",".join(key_cols)
    out["key_value"] = out[key_cols].astype(str).agg("|".join, axis=1)
    out = out.rename(columns={0: "count"})
    out.insert(0, "table", table)
    return out[["table","key_cols","key_value","count"]]


In [10]:
# write out

def write_excel_no_sci(df, path, sheet_name, float_format="0.######"):
    """
    note: Writes a DataFrame to Excel and forces numeric columns to use a fixed
    decimal format (no scientific notation). Replaces the target sheet when it exists.
    Uses a safe mkdir pattern when the path has no directory component.
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

    file_exists = os.path.exists(path)
    mode = "a" if file_exists else "w"
    writer_kwargs = dict(engine="openpyxl", mode=mode)
    if file_exists:
        writer_kwargs["if_sheet_exists"] = "replace"

    with pd.ExcelWriter(path, **writer_kwargs) as writer:
        df.to_excel(writer, index=False, sheet_name=sheet_name)
        ws = writer.sheets[sheet_name]

        for idx, col in enumerate(df.columns, start=1):
            if pd.api.types.is_numeric_dtype(df[col]):
                col_letter = get_column_letter(idx)
                for cell in ws[col_letter]:
                    cell.number_format = float_format

def write_dataframe_at(
    ws,
    df: pd.DataFrame,
    start_row: int,
    start_col: int,
    *,
    float_format: str = "0.######",
    include_header: bool = True,
) -> Tuple[int, int]:
    """
    note: Writes a DataFrame into an existing openpyxl worksheet at (start_row, start_col).
    Applies number_format to numeric cells. Returns (end_row, end_col) occupied by the write
    """
    if df is None or (isinstance(df, pd.DataFrame) and df.empty):
        return start_row - 1, start_col - 1

    cur_row = start_row
    rows = dataframe_to_rows(df, index=False, header=include_header)
    max_width = 0

    for r in rows:
        max_width = max(max_width, len(r))
        for j, v in enumerate(r, start=0):
            cell = ws.cell(row=cur_row, column=start_col + j, value=v)
            if isinstance(v, (int, float, np.integer, np.floating)) and v is not None:
                cell.number_format = float_format
        cur_row += 1

    end_row = cur_row - 1
    end_col = start_col + max_width - 1
    return end_row, end_col

def write_excel_blocks_left_to_right_one_sheet(
    blocks: List[Tuple[str, pd.DataFrame]],
    path: str,
    sheet_name: str,
    *,
    float_format: str = "0.######",
    blank_cols_between: int = 1,
    start_row: int = 1,
    start_col: int = 1,
    replace_sheet: bool = True,
) -> Tuple[int, int]:
    """
    note: Writes multiple (title, DataFrame) blocks left-to-right across a single worksheet.
    Returns (max_end_row, max_end_col) used by any block
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if os.path.exists(path):
        wb = load_workbook(path)
    else:
        wb = Workbook()

    if sheet_name in wb.sheetnames and replace_sheet:
        wb.remove(wb[sheet_name])
    ws = wb.create_sheet(title=sheet_name) if sheet_name not in wb.sheetnames else wb[sheet_name]

    cur_col = start_col
    max_end_row = start_row - 1
    max_end_col = start_col - 1

    for title, df in blocks:
        if df is None:
            continue
        if isinstance(df, pd.DataFrame) and df.empty:
            continue

        ws.cell(row=start_row, column=cur_col, value=str(title))
        end_row, end_col = write_dataframe_at(
            ws,
            df,
            start_row=start_row + 1,
            start_col=cur_col,
            float_format=float_format,
            include_header=True,
        )
        max_end_row = max(max_end_row, end_row)
        max_end_col = max(max_end_col, end_col)
        cur_col = end_col + blank_cols_between + 1

    wb.save(path)
    return max_end_row, max_end_col

def write_excel_blocks_top_to_bottom_one_sheet(
    blocks: List[Tuple[str, pd.DataFrame]],
    path: str,
    sheet_name: str,
    *,
    float_format: str = "0.######",
    blank_rows_between: int = 2,
    start_row: int = 1,
    start_col: int = 1,
    replace_sheet: bool = True,
) -> Tuple[int, int]:
    """
    note: Writes multiple (title, DataFrame) blocks top-to-bottom in a single worksheet.
    Returns (max_end_row, max_end_col) used by any block
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    if os.path.exists(path):
        wb = load_workbook(path)
    else:
        wb = Workbook()

    if sheet_name in wb.sheetnames and replace_sheet:
        wb.remove(wb[sheet_name])
    ws = wb.create_sheet(title=sheet_name) if sheet_name not in wb.sheetnames else wb[sheet_name]

    cur_row = start_row
    max_end_row = start_row - 1
    max_end_col = start_col - 1

    for title, df in blocks:
        if df is None:
            continue
        if isinstance(df, pd.DataFrame) and df.empty:
            continue

        ws.cell(row=cur_row, column=start_col, value=str(title))
        end_row, end_col = write_dataframe_at(
            ws,
            df,
            start_row=cur_row + 1,
            start_col=start_col,
            float_format=float_format,
            include_header=True,
        )
        max_end_row = max(max_end_row, end_row)
        max_end_col = max(max_end_col, end_col)
        cur_row = end_row + blank_rows_between + 1

    wb.save(path)
    return max_end_row, max_end_col

def write_section_blocks_wrapped_to_Z(
    ws,
    sections: list[tuple[str, pd.DataFrame]],
    *,
    start_row: int,
    start_col: int = 1,
    max_excel_col: int = 26,  # Z
    blank_cols_between: int = 1,
    blank_rows_between: int = 2,
    float_format: str = "0.######",
) -> int:
    """
    note: Writes multiple titled DataFrame sections left-to-right, wrapping to the next row band
    when the next section would exceed max_excel_col. Returns the next available row after all writes.
    """
    cur_row = start_row
    cur_col = start_col
    band_max_end_row = start_row - 1

    for title, df in sections:
        if df is None or (isinstance(df, pd.DataFrame) and df.empty):
            continue

        df_width = int(df.shape[1]) if isinstance(df, pd.DataFrame) else 1
        required_width = max(1, df_width)  # df columns
        required_end_col = cur_col + required_width - 1

        # If this block would exceed Z, wrap to next band (unless we're at band start)
        if required_end_col > max_excel_col and cur_col != start_col:
            cur_row = band_max_end_row + blank_rows_between + 1
            cur_col = start_col
            band_max_end_row = cur_row - 1
            required_end_col = cur_col + required_width - 1

        # Title row
        ws.cell(row=cur_row, column=cur_col, value=str(title))

        # DataFrame below title
        end_row, end_col = write_dataframe_at(
            ws,
            df,
            start_row=cur_row + 1,
            start_col=cur_col,
            float_format=float_format,
            include_header=True,
        )

        band_max_end_row = max(band_max_end_row, end_row)
        cur_col = int(end_col) + blank_cols_between + 1

    return int(band_max_end_row) + blank_rows_between + 1

def write_clean_csv_outputs(
    dfs_clean: Dict[str, pd.DataFrame],
    out_dir: str,
    *,
    suffix: str = "_clean",
) -> None:
    """
    note: Stage B exporter. Writes one cleaned CSV per table into out_dir.
    Assumes dfs_clean already includes any schema hygiene transformations
    """
    os.makedirs(out_dir, exist_ok=True)
    for t, df in dfs_clean.items():
        out_path = os.path.join(out_dir, f"{t}{suffix}.csv")
        df.to_csv(out_path, index=False)

def write_auto_audit_sheet(
    audit_path: str,
    auto_summary: pd.DataFrame,
    auto_details: pd.DataFrame,
    ingestion_exceptions: pd.DataFrame | None = None,
    split_audit_df: pd.DataFrame | None = None,
    *,
    sheet_name: str = "auto_audit",
) -> None:
    """
    note: Writes/overwrites the auto_audit sheet with proof of applied changes plus guardrail outcomes and optional delimiter split audit.
    """
    blocks = [("auto_summary", auto_summary)]

    max_end_row, _ = write_excel_blocks_left_to_right_one_sheet(
        blocks,
        path=audit_path,
        sheet_name=sheet_name,
        blank_cols_between=1,
        start_row=1,
        start_col=1,
        replace_sheet=True,
    )

    wb = load_workbook(audit_path)
    ws = wb[sheet_name]

    sections_start_row = int(max_end_row) + 3 if int(max_end_row) > 0 else 1
    sections = [("rename_details_where_changed", auto_details)]

    if ingestion_exceptions is not None:
        sections.append(("ingestion_exceptions", ingestion_exceptions))

    if split_audit_df is not None:
        sections.append(("split_audit", split_audit_df))

    _ = write_section_blocks_wrapped_to_Z(
        ws,
        sections,
        start_row=sections_start_row,
        start_col=1,
        max_excel_col=26,
        blank_cols_between=1,
        blank_rows_between=2,
        float_format="0.######",
    )

    wb.save(audit_path)


In [11]:
# data dictionary

def build_generated_data_dictionary(table_name: str, df: pd.DataFrame, *, sample_values: int = 3, stage: str = "") -> pd.DataFrame:
    """
    note: Builds a self-contained generated data dictionary from observed data.
    Includes a few sample non-null distinct values for quick review.
    """
    n = len(df)
    rows: list[dict] = []
    for c in df.columns:
        s = df[c]
        nn = int(s.notna().sum())
        nun = int(s.nunique(dropna=True))
        null_rate = 1.0 - (nn / n) if n else np.nan
        unique_ratio = (nun / nn) if nn else np.nan

        samples: list[str] = []
        if nn > 0:
            ss = s.dropna()
            try:
                uniq = pd.unique(ss)
                samples = [str(x) for x in uniq[:sample_values]]
            except Exception:
                samples = [str(x) for x in ss.head(sample_values).tolist()]

        rows.append(
            {
                "stage": stage,
                "table": table_name,
                "column": c,
                "dtype": str(s.dtype),
                "n_rows": int(n),
                "n_nonnull": nn,
                "n_unique": nun,
                "null_rate": float(null_rate) if pd.notna(null_rate) else np.nan,
                "unique_ratio": float(unique_ratio) if pd.notna(unique_ratio) else np.nan,
                "sample_values": " | ".join(samples),
            }
        )
    return pd.DataFrame(rows)


In [12]:
# auditing

def build_clean_plan(
    dropped_cols_all: pd.DataFrame,
    collisions_all: pd.DataFrame,
    delimited_candidates_all: pd.DataFrame,
    *,
    include_dedup_placeholders: bool = True,
) -> pd.DataFrame:
    """
    note: Produces a minimal plan/record for the cleaning stage suitable for human review.
    Includes delimiter-split and (optionally) dedup placeholders as pending steps.
    """
    n_dropped = 0 if dropped_cols_all is None or dropped_cols_all.empty else int(len(dropped_cols_all))
    n_collisions = 0 if collisions_all is None or collisions_all.empty else int(len(collisions_all))
    n_delim_cands = 0 if delimited_candidates_all is None or delimited_candidates_all.empty else int(len(delimited_candidates_all))

    rows = [
        {
            "step": 1,
            "operation": "drop_pandas_index_artifacts",
            "status": "applied",
            "detail": "Drops columns starting with 'Unnamed' (case-insensitive).",
            "review": f"See section 'dropped_columns' on clean_audit. Rows: {n_dropped}",
        },
        {
            "step": 2,
            "operation": "normalize_columns_simple",
            "status": "applied",
            "detail": "strip leading/trailing spaces; lowercase; whitespace->underscore. No prefixing.",
            "review": "See 'column_map' on clean_audit (raw_column -> final_column).",
        },
        {
            "step": 3,
            "operation": "resolve_name_collisions",
            "status": "applied_if_needed",
            "detail": "If normalization causes collisions, suffix _2, _3, ... deterministically.",
            "review": f"See section 'name_collisions' on clean_audit. Rows: {n_collisions}",
        },
        {
            "step": 4,
            "operation": "detect_delimited_columns",
            "status": "applied",
            "detail": "Detects likely delimited text columns for review (no transform).",
            "review": f"See sections 'delimited_candidates' and 'delimited_samples'. Candidates: {n_delim_cands}",
        },
        {
            "step": 5,
            "operation": "detect_sparse_projection_dups",
            "status": "applied",
            "detail": "Audit-only: checks exact duplicates on dense-column projection (null_rate <= threshold). No value normalization. No rows dropped.",
            "review": "See sections 'sparse_projection_dup_summary' and 'sparse_projection_dup_samples' on clean_audit.",
        },
        {
            "step": 6,
            "operation": "split_delimited_columns",
            "status": "pending",
            "detail": "No auto-splitting by default. Apply only via explicit split_allowlist.",
            "review": "If approved, post-clean allowlists cell will write split_audit into auto_audit.",
        },
        {
            "step": 7,
            "operation": "audit_gate",
            "status": "pending",
            "detail": "User reviews audit workbook before applying optional allowlists and writing cleaned CSVs.",
            "review": "Review clean_audit, then run the post-clean allowlists cell.",
        },
    ]

    if include_dedup_placeholders:
        rows.append(
            {
                "step": 8,
                "operation": "dedup_detection_and_review",
                "status": "pending",
                "detail": "Dedup is not automatic. Detection is audit-only; dropping rows requires explicit allowlists.",
                "review": "Run dedup_audit generation (dedup_audit/dedup_plan), then decide allowlists.",
            }
        )
        rows.append(
            {
                "step": 9,
                "operation": "apply_dedup_allowlists",
                "status": "pending",
                "detail": "Apply exact-row and/or key-based dedup only via explicit allowlists.",
                "review": "If approved, post-clean allowlists cell will write dedup_apply_audit and update auto_audit.",
            }
        )

    return pd.DataFrame(rows)

def build_clean_audit_workbook(
    dfs_raw: dict[str, pd.DataFrame],
    *,
    audit_path: str = AUDIT_XLSX,
    max_candidate_keys_per_table: int = 25,
    ingestion_exceptions: pd.DataFrame | None = None,
) -> dict[str, object]:
    """
    note: Stage A builder. Constructs cleaned dfs in-memory (schema hygiene only),
    builds audit artifacts, writes only these sheets:
    - clean_audit (top blocks + wrapped section blocks left-to-right until col Z)
    - clean_plan
    - data_dictionary (written raw first, then overwritten with clean)
    Returns artifacts for Stage B export.
    """
    # Pre-clean data_dictionary (raw)
    dd_raw_blocks: list[pd.DataFrame] = []
    for t, df in dfs_raw.items():
        dd_raw_blocks.append(build_generated_data_dictionary(t, df, sample_values=3, stage="raw"))
    dd_raw_all = pd.concat(dd_raw_blocks, ignore_index=True) if dd_raw_blocks else pd.DataFrame(
        columns=["stage","table","column","dtype","n_rows","n_nonnull","n_unique","null_rate","unique_ratio","sample_values"]
    )
    write_excel_no_sci(dd_raw_all, audit_path, "data_dictionary")

    # Build cleaned dfs + audit tables
    dfs_clean: dict[str, pd.DataFrame] = {}

    col_maps: list[pd.DataFrame] = []
    collisions_logs: list[pd.DataFrame] = []
    profiles: list[pd.DataFrame] = []
    cand_keys_blocks: list[pd.DataFrame] = []
    dup_sigs: list[pd.DataFrame] = []
    dropped_cols_logs: list[pd.DataFrame] = []
    delimited_candidates_logs: list[pd.DataFrame] = []
    delimited_samples_logs: list[pd.DataFrame] = []

    for t, df in dfs_raw.items():
        df0, dropped = drop_pandas_index_artifacts(df)
        if dropped is not None and not dropped.empty:
            dropped = dropped.copy()
            dropped.insert(0, "table", t)
            dropped_cols_logs.append(dropped)

        col_map, collisions = build_column_map(t, df0)
        df1 = apply_column_map(df0, col_map)

        cands_delim = detect_delimited_columns(df1)
        if cands_delim is not None and not cands_delim.empty:
            cands_delim = cands_delim.copy()
            cands_delim.insert(0, "table", t)
            delimited_candidates_logs.append(cands_delim)
        
            # sample values for the top few candidates per table
            topk = cands_delim.head(5)
            for _, row in topk.iterrows():
                samp = sample_delimited_values(df1, row["column"], row["delimiter"], n=10)
                if samp is not None and not samp.empty:
                    samp = samp.copy()
                    samp.insert(0, "table", t)
                    delimited_samples_logs.append(samp)
        
        dfs_clean[t] = df1
        col_maps.append(col_map)

        if collisions is not None and not collisions.empty:
            collisions = collisions.copy()
            collisions.insert(0, "table", t)
            collisions_logs.append(collisions)

        prof = profile_table(t, df1)
        cands = candidate_identity_keys(prof).head(int(max_candidate_keys_per_table)).copy()
        cands.insert(0, "rank", np.arange(1, len(cands) + 1))

        sig = duplicate_signature(t, df1, key_cols=list(cands["column"]))

        profiles.append(prof)
        cand_keys_blocks.append(cands)
        dup_sigs.append(sig)

    col_map_all = pd.concat(col_maps, ignore_index=True) if col_maps else pd.DataFrame()
    collisions_all = pd.concat(collisions_logs, ignore_index=True) if collisions_logs else pd.DataFrame()
    profile_all = pd.concat(profiles, ignore_index=True) if profiles else pd.DataFrame()
    cand_keys_all = pd.concat(cand_keys_blocks, ignore_index=True) if cand_keys_blocks else pd.DataFrame()
    dup_sig_all = pd.concat(dup_sigs, ignore_index=True) if dup_sigs else pd.DataFrame()
    dropped_cols_all = pd.concat(dropped_cols_logs, ignore_index=True) if dropped_cols_logs else pd.DataFrame()
    delimited_candidates_all = pd.concat(delimited_candidates_logs, ignore_index=True) if delimited_candidates_logs else pd.DataFrame(
        columns=["table","column","dtype","nonnull","unique_ratio","delimiter","delim_hit_rate","mean_delims_when_hit"]
    )
    delimited_samples_all = pd.concat(delimited_samples_logs, ignore_index=True) if delimited_samples_logs else pd.DataFrame(
        columns=["table","column","delimiter","value"]
    )

    delimited_candidates_all = (
        pd.concat(delimited_candidates_logs, ignore_index=True)
        if delimited_candidates_logs
        else _empty_delimited_candidates_df()
    )
    
    delimited_samples_all = (
        pd.concat(delimited_samples_logs, ignore_index=True)
        if delimited_samples_logs
        else _empty_delimited_samples_df()
    )

    dedup_summary_all = build_dedup_summary_table(dfs_clean)
    split_allowlist_template = build_split_allowlist_template(delimited_candidates_all)

    sparse_proj_summary_all, sparse_proj_samples_all = build_sparse_projection_dup_audit(
        dfs_clean,
        null_rate_threshold=0.05,
        max_dense_cols=50,
        sample_groups=15,
    )

    # Summary blocks for top of clean_audit
    shapes_rows = [{"table": t, "n_rows": int(df.shape[0]), "n_cols": int(df.shape[1])} for t, df in dfs_clean.items()]
    shapes_df = (
        pd.DataFrame(shapes_rows)
        .sort_values(["n_rows", "n_cols"], ascending=[False, False])
        .reset_index(drop=True)
    )

    ren_sum = (
        col_map_all.groupby(["table"], as_index=False)
        .agg(n_cols=("raw_column", "count"))
        .sort_values(["n_cols"], ascending=False)
        .reset_index(drop=True)
    )

    if not collisions_all.empty:
        c = collisions_all.groupby(["table"], as_index=False).agg(n_collisions=("collision_n", "count"))
        ren_sum = ren_sum.merge(c, on="table", how="left")

    if not dropped_cols_all.empty:
        d = dropped_cols_all.groupby(["table"], as_index=False).agg(n_dropped=("dropped_column", "count"))
        ren_sum = ren_sum.merge(d, on="table", how="left")

    if "n_collisions" not in ren_sum.columns:
        ren_sum["n_collisions"] = 0
    if "n_dropped" not in ren_sum.columns:
        ren_sum["n_dropped"] = 0

    ren_sum = ren_sum.fillna(0)

    # Write clean_audit: top blocks (left-to-right, 1 blank col between)
    top_blocks = [
        ("table_shapes", shapes_df),
        ("normalize_summary", ren_sum),
        ("candidate_keys_top", cand_keys_all),
        ("dup_signatures", dup_sig_all),
    ]
    max_end_row, _ = write_excel_blocks_left_to_right_one_sheet(
        top_blocks,
        path=audit_path,
        sheet_name="clean_audit",
        blank_cols_between=1,
        start_row=1,
        start_col=1,
        replace_sheet=True,
    )

    # Below: write remaining sections wrapped left-to-right until col Z, then wrap
    wb = load_workbook(audit_path)
    ws = wb["clean_audit"]

    sections_start_row = int(max_end_row) + 3 if int(max_end_row) > 0 else 1

    # Ensure ingestion_exceptions is always visible
    if ingestion_exceptions is None:
        ingestion_exceptions = pd.DataFrame(
            [{
                "file": "",
                "file_type": "",
                "sheet": "",
                "status": "ok",
                "reason": "ingestion_guardrail_not_run_or_not_provided",
                "details": "",
            }],
            columns=["file","file_type","sheet","status","reason","details"],
        )
    elif ingestion_exceptions.empty:
        ingestion_exceptions = pd.DataFrame(
            [{
                "file": "",
                "file_type": "",
                "sheet": "",
                "status": "ok",
                "reason": "no_exceptions_detected",
                "details": "",
            }],
            columns=["file","file_type","sheet","status","reason","details"],
        )

    sections = [
        ("ingestion_exceptions", ingestion_exceptions),
    
        ("dedup_summary", dedup_summary_all),
        ("sparse_projection_dup_summary", sparse_proj_summary_all),
        ("sparse_projection_dup_samples", sparse_proj_samples_all),
    
        ("delimited_candidates", delimited_candidates_all),
        ("delimited_samples", delimited_samples_all),
        ("split_allowlist_template", split_allowlist_template),
        ("column_map", col_map_all),
        ("name_collisions", collisions_all if not collisions_all.empty else pd.DataFrame(columns=["table","original_normalized","resolved_name","collision_n"])),
        ("dropped_columns", dropped_cols_all if not dropped_cols_all.empty else pd.DataFrame(columns=["table","dropped_column"])),
        ("column_profiles", profile_all),
    ]

    _ = write_section_blocks_wrapped_to_Z(
        ws,
        sections,
        start_row=sections_start_row,
        start_col=1,
        max_excel_col=26,
        blank_cols_between=1,
        blank_rows_between=2,
        float_format="0.######",
    )

    wb.save(audit_path)

    # clean_plan tab
    plan_df = build_clean_plan(dropped_cols_all, collisions_all, delimited_candidates_all, include_dedup_placeholders=True)
    write_excel_no_sci(plan_df, audit_path, "clean_plan")

    # Overwrite data_dictionary with post-clean view (clean)
    dd_clean_blocks: list[pd.DataFrame] = []
    for t, df in dfs_clean.items():
        dd_clean_blocks.append(build_generated_data_dictionary(t, df, sample_values=3, stage="clean"))
    dd_clean_all = pd.concat(dd_clean_blocks, ignore_index=True) if dd_clean_blocks else pd.DataFrame(
        columns=["stage","table","column","dtype","n_rows","n_nonnull","n_unique","null_rate","unique_ratio","sample_values"]
    )
    write_excel_no_sci(dd_clean_all, audit_path, "data_dictionary")

    return {
        "dfs_clean": dfs_clean,
        "col_map_all": col_map_all,
        "profile_all": profile_all,
        "cand_keys_all": cand_keys_all,
        "dup_sig_all": dup_sig_all,
        "collisions_all": collisions_all,
        "dropped_cols_all": dropped_cols_all,
        "audit_path": audit_path,
        "ingestion_exceptions": ingestion_exceptions,
    }

def build_dedup_plan(
    exact_dup_summary_all: pd.DataFrame,
    key_dup_summary_all: pd.DataFrame,
) -> pd.DataFrame:
    """
    note: Produces a minimal dedup plan for human review.
    No actions are applied automatically; allowlists are required.
    """
    n_exact_tables = 0 if exact_dup_summary_all is None or exact_dup_summary_all.empty else int((exact_dup_summary_all["dup_rows"] > 0).sum())
    n_key_tables = 0 if key_dup_summary_all is None or key_dup_summary_all.empty else int((key_dup_summary_all["dup_rows_sum_across_candidates"] > 0).sum())

    rows = [
        {
            "step": 1,
            "operation": "detect_exact_row_duplicates",
            "status": "applied",
            "detail": "Detect exact row duplicates via stable row hashing (no value normalization).",
            "review": f"See dedup_audit exact_row_dup_summary/sample. Tables with dup rows: {n_exact_tables}",
        },
        {
            "step": 2,
            "operation": "detect_key_duplicates_for_candidates",
            "status": "applied",
            "detail": "For candidate key columns (from profile-based selection), compute dup_keys/dup_rows.",
            "review": f"See dedup_audit key_dup_summary and offenders. Tables with dup rows: {n_key_tables}",
        },
        {
            "step": 3,
            "operation": "apply_exact_row_dedup_allowlist",
            "status": "pending",
            "detail": "Drop exact duplicate rows only for tables in explicit allowlist.",
            "review": "If approved, Stage D apply will write dedup_apply_audit into auto_audit.",
        },
        {
            "step": 4,
            "operation": "apply_key_dedup_allowlist",
            "status": "pending",
            "detail": "Deduplicate on explicit key_cols with deterministic keep rule only via allowlist.",
            "review": "If approved, Stage D apply will write dedup_apply_audit into auto_audit.",
        },
    ]
    return pd.DataFrame(rows)

def build_dedup_summary_table(dfs_clean: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    note: Builds an always-present dedup summary per table.
    Uses exact-row duplication only (no value normalization). Always returns >= 1 row.
    """
    rows = []
    for t, df in dfs_clean.items():
        n = int(df.shape[0])
        if n == 0:
            rows.append({"table": t, "n_rows": 0, "exact_dup_rows": 0, "exact_dup_groups": 0, "exact_dup_row_rate": np.nan})
            continue

        # exact duplicates across all columns
        dup_mask = df.duplicated(keep=False)
        exact_dup_rows = int(dup_mask.sum())

        # approximate group count via counting duplicates after first occurrence
        exact_dup_groups = int(df.duplicated(keep="first").sum())

        rows.append(
            {
                "table": t,
                "n_rows": n,
                "exact_dup_rows": exact_dup_rows,
                "exact_dup_groups": exact_dup_groups,
                "exact_dup_row_rate": float(exact_dup_rows / n) if n else np.nan,
            }
        )

    out = pd.DataFrame(rows).sort_values(["exact_dup_rows", "n_rows"], ascending=[False, False]).reset_index(drop=True)

    if out.empty:
        out = pd.DataFrame(
            [{
                "table": "",
                "n_rows": 0,
                "exact_dup_rows": 0,
                "exact_dup_groups": 0,
                "exact_dup_row_rate": np.nan,
                "status": "no_tables",
            }]
        )

    return out


In [13]:
# applying auto clean

def apply_clean_plan(
    dfs_raw: dict[str, pd.DataFrame],
    *,
    split_allowlist: list[dict] | None = None,
) -> tuple[dict[str, pd.DataFrame], pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    note: Applies automated clean steps deterministically to dfs_raw and optionally applies user-approved delimiter splits.
    Returns (dfs_clean, col_map_all, collisions_all, dropped_cols_all, split_audit_df).
    """
    dfs_clean: dict[str, pd.DataFrame] = {}

    col_maps: list[pd.DataFrame] = []
    collisions_logs: list[pd.DataFrame] = []
    dropped_cols_logs: list[pd.DataFrame] = []

    for t, df in dfs_raw.items():
        df0, dropped = drop_pandas_index_artifacts(df)
        if dropped is not None and not dropped.empty:
            dropped = dropped.copy()
            dropped.insert(0, "table", t)
            dropped_cols_logs.append(dropped)

        col_map, collisions = build_column_map(t, df0)
        df1 = apply_column_map(df0, col_map)

        dfs_clean[t] = df1
        col_maps.append(col_map)

        if collisions is not None and not collisions.empty:
            collisions = collisions.copy()
            collisions.insert(0, "table", t)
            collisions_logs.append(collisions)

    col_map_all = pd.concat(col_maps, ignore_index=True) if col_maps else pd.DataFrame()
    collisions_all = pd.concat(collisions_logs, ignore_index=True) if collisions_logs else pd.DataFrame()
    dropped_cols_all = pd.concat(dropped_cols_logs, ignore_index=True) if dropped_cols_logs else pd.DataFrame()

    # Optional delimiter splits (explicit allowlist only)
    dfs_clean2, split_audit_df = apply_delimited_splits(dfs_clean, split_allowlist)

    return dfs_clean2, col_map_all, collisions_all, dropped_cols_all, split_audit_df

def apply_delimited_splits(
    dfs_clean: dict[str, pd.DataFrame],
    split_allowlist: list[dict] | None,
) -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    """
    note: Applies user-approved delimiter splits to dfs_clean.
    split_allowlist items:
      {"table": "...", "column": "...", "delimiter": ";", "max_splits": 4, "keep_original": True, "strip_tokens": True}
    Returns (dfs_clean_out, split_audit_df) where split_audit_df records what was done.
    """
    if not split_allowlist:
        return dfs_clean, pd.DataFrame(columns=["table","column","delimiter","max_splits","keep_original","strip_tokens","new_columns_added"])

    out: dict[str, pd.DataFrame] = {k: v.copy() for k, v in dfs_clean.items()}
    rows: list[dict] = []

    for item in split_allowlist:
        t = item.get("table", "")
        c = item.get("column", "")
        d = item.get("delimiter", "")

        if t not in out:
            rows.append({"table": t, "column": c, "delimiter": d, "max_splits": np.nan, "keep_original": np.nan, "strip_tokens": np.nan, "new_columns_added": "skipped_table_missing"})
            continue
        if c not in out[t].columns:
            rows.append({"table": t, "column": c, "delimiter": d, "max_splits": np.nan, "keep_original": np.nan, "strip_tokens": np.nan, "new_columns_added": "skipped_column_missing"})
            continue
        if not isinstance(d, str) or d == "":
            rows.append({"table": t, "column": c, "delimiter": d, "max_splits": np.nan, "keep_original": np.nan, "strip_tokens": np.nan, "new_columns_added": "skipped_bad_delimiter"})
            continue

        max_splits = int(item.get("max_splits", 4))
        keep_original = bool(item.get("keep_original", True))
        strip_tokens = bool(item.get("strip_tokens", True))

        before_cols = set(out[t].columns)
        out[t] = split_delimited_column(
            out[t],
            c,
            d,
            max_splits=max_splits,
            keep_original=keep_original,
            strip_tokens=strip_tokens,
        )
        after_cols = set(out[t].columns)
        added = sorted(list(after_cols - before_cols))
        rows.append(
            {
                "table": t,
                "column": c,
                "delimiter": d,
                "max_splits": max_splits,
                "keep_original": keep_original,
                "strip_tokens": strip_tokens,
                "new_columns_added": ", ".join(added) if added else "",
            }
        )

    return out, pd.DataFrame(rows)

def apply_dedup_allowlists(
    dfs_in: dict[str, pd.DataFrame],
    *,
    exact_row_tables: list[str] | None = None,
    key_rules: list[dict] | None = None,
) -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    """
    note: Applies dedup actions only via explicit allowlists.
    exact_row_tables: list of table names to drop exact duplicate rows.
    key_rules items:
      {
        "table": "dim_customer",
        "key_cols": ["customer_ref_id"],
        "keep": "first" | "last" | "max_nonnull"
      }
    Returns (dfs_out, dedup_apply_audit_df).
    """
    exact_row_tables = exact_row_tables or []
    key_rules = key_rules or []

    out = {k: v.copy() for k, v in dfs_in.items()}
    rows: list[dict] = []

    # 1) exact row dedup
    for t in exact_row_tables:
        if t not in out:
            rows.append({"table": t, "mode": "exact_row", "status": "skipped_table_missing", "rows_before": np.nan, "rows_after": np.nan, "rows_removed": np.nan, "detail": ""})
            continue
        df = out[t]
        before = int(df.shape[0])
        df2 = df.drop_duplicates(keep="first")
        after = int(df2.shape[0])
        out[t] = df2
        rows.append({"table": t, "mode": "exact_row", "status": "applied", "rows_before": before, "rows_after": after, "rows_removed": before - after, "detail": ""})

    # 2) key-based dedup
    for rule in key_rules:
        t = rule.get("table", "")
        key_cols = rule.get("key_cols", [])
        keep = rule.get("keep", "first")

        if t not in out:
            rows.append({"table": t, "mode": "key", "status": "skipped_table_missing", "rows_before": np.nan, "rows_after": np.nan, "rows_removed": np.nan, "detail": ""})
            continue
        if not key_cols:
            rows.append({"table": t, "mode": "key", "status": "skipped_no_key_cols", "rows_before": np.nan, "rows_after": np.nan, "rows_removed": np.nan, "detail": ""})
            continue
        missing = [c for c in key_cols if c not in out[t].columns]
        if missing:
            rows.append({"table": t, "mode": "key", "status": "skipped_missing_key_cols", "rows_before": np.nan, "rows_after": np.nan, "rows_removed": np.nan, "detail": f"missing={','.join(missing)}"})
            continue

        df = out[t]
        before = int(df.shape[0])

        if keep in ("first", "last"):
            df2 = df.drop_duplicates(subset=key_cols, keep=keep)
        elif keep == "max_nonnull":
            nn = df.notna().sum(axis=1)
            tmp = df.copy()
            tmp["_nonnull_score"] = nn
            tmp["_row_order"] = np.arange(len(tmp))
            tmp = tmp.sort_values(by=key_cols + ["_nonnull_score", "_row_order"], ascending=[True]*len(key_cols) + [False, True])
            df2 = tmp.drop_duplicates(subset=key_cols, keep="first").drop(columns=["_nonnull_score", "_row_order"])
        else:
            rows.append({"table": t, "mode": "key", "status": "skipped_bad_keep_rule", "rows_before": before, "rows_after": before, "rows_removed": 0, "detail": f"keep={keep}"})
            continue

        after = int(df2.shape[0])
        out[t] = df2
        rows.append({"table": t, "mode": "key", "status": "applied", "rows_before": before, "rows_after": after, "rows_removed": before - after, "detail": f"key_cols={','.join(key_cols)} keep={keep}"})

    audit = pd.DataFrame(rows, columns=["table","mode","status","rows_before","rows_after","rows_removed","detail"])
    return out, audit


In [14]:
def build_auto_audit(
    dfs_raw: dict[str, pd.DataFrame],
    dfs_clean: dict[str, pd.DataFrame],
    col_map_all: pd.DataFrame,
    collisions_all: pd.DataFrame,
    dropped_cols_all: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    note: Builds audit tables proving that automated steps were applied.
    Returns (auto_summary_df, auto_details_df) for writing to the auto_audit sheet.
    """
    rows = []
    for t, raw_df in dfs_raw.items():
        clean_df = dfs_clean.get(t)
        if clean_df is None:
            continue

        raw_cols = set(map(str, raw_df.columns))
        clean_cols = set(map(str, clean_df.columns))

        rows.append(
            {
                "table": t,
                "raw_rows": int(raw_df.shape[0]),
                "clean_rows": int(clean_df.shape[0]),
                "raw_cols": int(raw_df.shape[1]),
                "clean_cols": int(clean_df.shape[1]),
                "rows_changed": int(raw_df.shape[0]) != int(clean_df.shape[0]),
                "cols_changed": int(raw_df.shape[1]) != int(clean_df.shape[1]) or raw_cols != clean_cols,
                "n_cols_dropped": int(dropped_cols_all[dropped_cols_all["table"] == t].shape[0]) if not dropped_cols_all.empty else 0,
                "n_name_collisions": int(collisions_all[collisions_all["table"] == t].shape[0]) if not collisions_all.empty else 0,
            }
        )

    auto_summary = pd.DataFrame(rows).sort_values(["table"]).reset_index(drop=True)

    # details: show mapping deltas only (where raw_column != final_column)
    if col_map_all is None or col_map_all.empty:
        auto_details = pd.DataFrame(columns=["table", "raw_column", "final_column"])
    else:
        auto_details = col_map_all.loc[col_map_all["raw_column"].astype(str) != col_map_all["final_column"].astype(str), ["table", "raw_column", "final_column"]].copy()
        auto_details = auto_details.sort_values(["table", "raw_column"]).reset_index(drop=True)

    return auto_summary, auto_details

def build_dedup_audit_workbook(
    dfs_clean: dict[str, pd.DataFrame],
    *,
    audit_path: str,
    max_candidate_keys_per_table: int = 25,
) -> dict[str, pd.DataFrame]:
    """
    note: Builds dedup detection artifacts (no data changes) and writes:
    - dedup_audit
    - dedup_plan
    Returns dict of dedup artifacts for later apply/audit steps.
    """
    exact_summaries: list[pd.DataFrame] = []
    exact_samples: list[pd.DataFrame] = []

    key_summ_rows: list[dict] = []
    key_offenders: list[pd.DataFrame] = []

    for t, df in dfs_clean.items():
        s, samp = exact_row_duplicate_summary(t, df, sample_groups=10)
        exact_summaries.append(s)
        if samp is not None and not samp.empty:
            exact_samples.append(samp)

        # candidate keys based on profile_table + candidate_identity_keys already in your notebook
        prof = profile_table(t, df)
        cands = candidate_identity_keys(prof).head(int(max_candidate_keys_per_table)).copy()
        key_cols = list(cands["column"]) if not cands.empty else []

        # summarize per candidate column
        sig = duplicate_signature(t, df, key_cols=key_cols)
        if sig is not None and not sig.empty:
            # roll up for plan metrics: any dup rows at all for any candidate col
            dup_rows_any = int(sig["dup_rows"].fillna(0).sum())
            dup_keys_any = int(sig["dup_keys"].fillna(0).sum())
        else:
            dup_rows_any = 0
            dup_keys_any = 0

        key_summ_rows.append({
            "table": t,
            "n_candidate_key_cols": int(len(key_cols)),
            "dup_rows_sum_across_candidates": int(dup_rows_any),
            "dup_keys_sum_across_candidates": int(dup_keys_any),
        })

        # offenders: only for candidates that actually show dup_keys > 0
        if sig is not None and not sig.empty:
            bad = sig[sig["dup_keys"].fillna(0) > 0].copy()
            for _, r in bad.iterrows():
                offenders = key_duplicate_offenders(t, df, [str(r["key_col"])], top_n=20)
                if offenders is not None and not offenders.empty:
                    key_offenders.append(offenders)

    exact_row_dup_summary_all = pd.concat(exact_summaries, ignore_index=True) if exact_summaries else pd.DataFrame(
        columns=["table","n_rows","dup_rows","dup_groups","dup_row_rate"]
    )
    exact_row_dup_samples_all = pd.concat(exact_samples, ignore_index=True) if exact_samples else pd.DataFrame(
        columns=["table","row_hash","count"]
    )

    key_dup_summary_all = pd.DataFrame(key_summ_rows) if key_summ_rows else pd.DataFrame(
        columns=["table","n_candidate_key_cols","dup_rows_sum_across_candidates","dup_keys_sum_across_candidates"]
    )
    key_dup_offenders_all = pd.concat(key_offenders, ignore_index=True) if key_offenders else pd.DataFrame(
        columns=["table","key_cols","key_value","count"]
    )

    # write dedup_audit sheet
    blocks = [("exact_row_dup_summary", exact_row_dup_summary_all)]
    max_end_row, _ = write_excel_blocks_left_to_right_one_sheet(
        blocks,
        path=audit_path,
        sheet_name="dedup_audit",
        blank_cols_between=1,
        start_row=1,
        start_col=1,
        replace_sheet=True,
    )

    wb = load_workbook(audit_path)
    ws = wb["dedup_audit"]
    sections_start_row = int(max_end_row) + 3 if int(max_end_row) > 0 else 1

    sections = [
        ("exact_row_dup_samples", exact_row_dup_samples_all),
        ("key_dup_summary", key_dup_summary_all),
        ("key_dup_offenders", key_dup_offenders_all),
    ]
    _ = write_section_blocks_wrapped_to_Z(
        ws,
        sections,
        start_row=sections_start_row,
        start_col=1,
        max_excel_col=26,
        blank_cols_between=1,
        blank_rows_between=2,
        float_format="0.######",
    )
    wb.save(audit_path)

    # write dedup_plan
    dedup_plan_df = build_dedup_plan(exact_row_dup_summary_all, key_dup_summary_all)
    write_excel_no_sci(dedup_plan_df, audit_path, "dedup_plan")

    return {
        "exact_row_dup_summary_all": exact_row_dup_summary_all,
        "exact_row_dup_samples_all": exact_row_dup_samples_all,
        "key_dup_summary_all": key_dup_summary_all,
        "key_dup_offenders_all": key_dup_offenders_all,
        "dedup_plan_df": dedup_plan_df,
    }

def build_sparse_projection_dup_audit(
    dfs_clean: dict[str, pd.DataFrame],
    *,
    null_rate_threshold: float = 0.05,
    max_dense_cols: int = 50,
    sample_groups: int = 15,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    note: Audit-only detection of potential partial-duplicate rows by projecting each table onto
    "dense" columns (null_rate <= threshold) and checking for exact duplicates on that projection.
    No value normalization is performed. Returns (summary_df, sample_groups_df).
    """
    summary_rows: list[dict] = []
    sample_rows: list[pd.DataFrame] = []

    for t, df in dfs_clean.items():
        if df is None or df.empty:
            summary_rows.append({
                "table": t,
                "n_rows": 0,
                "n_cols": 0,
                "n_dense_cols_used": 0,
                "dense_cols_used": "",
                "null_rate_threshold": float(null_rate_threshold),
                "dup_rows": 0,
                "dup_groups": 0,
                "dup_row_rate": np.nan,
                "status": "empty_table",
            })
            continue

        n_rows = int(df.shape[0])
        n_cols = int(df.shape[1])

        # compute null rates (no value normalization)
        null_rates = df.isna().mean(numeric_only=False)
        dense_cols = [c for c in df.columns if float(null_rates.get(c, 1.0)) <= float(null_rate_threshold)]

        # cap to avoid very wide hashing
        if len(dense_cols) > int(max_dense_cols):
            dense_cols = dense_cols[:int(max_dense_cols)]

        if len(dense_cols) == 0:
            summary_rows.append({
                "table": t,
                "n_rows": n_rows,
                "n_cols": n_cols,
                "n_dense_cols_used": 0,
                "dense_cols_used": "",
                "null_rate_threshold": float(null_rate_threshold),
                "dup_rows": 0,
                "dup_groups": 0,
                "dup_row_rate": 0.0,
                "status": "no_dense_cols_under_threshold",
            })
            continue

        proj = df[dense_cols]
        row_hash = pd.util.hash_pandas_object(proj, index=False)
        vc = row_hash.value_counts()

        dup_groups = int((vc > 1).sum())
        dup_rows = int(vc[vc > 1].sum())
        dup_row_rate = float(dup_rows / n_rows) if n_rows else np.nan

        summary_rows.append({
            "table": t,
            "n_rows": n_rows,
            "n_cols": n_cols,
            "n_dense_cols_used": int(len(dense_cols)),
            "dense_cols_used": ",".join(map(str, dense_cols)),
            "null_rate_threshold": float(null_rate_threshold),
            "dup_rows": dup_rows,
            "dup_groups": dup_groups,
            "dup_row_rate": dup_row_rate if pd.notna(dup_row_rate) else np.nan,
            "status": "ok",
        })

        if dup_groups > 0:
            samp = vc[vc > 1].head(int(sample_groups)).reset_index()
            samp.columns = ["row_hash", "count"]
            samp.insert(0, "table", t)
            samp.insert(2, "n_dense_cols_used", int(len(dense_cols)))
            sample_rows.append(samp)

    summary_df = pd.DataFrame(summary_rows)
    if summary_df.empty:
        summary_df = pd.DataFrame([{
            "table": "",
            "n_rows": 0,
            "n_cols": 0,
            "n_dense_cols_used": 0,
            "dense_cols_used": "",
            "null_rate_threshold": float(null_rate_threshold),
            "dup_rows": 0,
            "dup_groups": 0,
            "dup_row_rate": np.nan,
            "status": "no_tables",
        }])

    summary_df = summary_df.sort_values(["dup_rows", "n_rows"], ascending=[False, False]).reset_index(drop=True)

    samples_df = pd.concat(sample_rows, ignore_index=True) if sample_rows else pd.DataFrame(
        [{
            "table": "",
            "row_hash": "",
            "n_dense_cols_used": 0,
            "count": 0,
            "status": "no_dup_groups_detected",
        }],
        columns=["table","row_hash","n_dense_cols_used","count","status"],
    )

    return summary_df, samples_df


# human in the loop - automated cleaning

In [15]:
# data dictionary runs

dd_blocks = []
for t, df in dfs_raw.items():
    dd_blocks.append(build_generated_data_dictionary(t, df, sample_values=3, stage="raw"))

dd_all_raw = pd.concat(dd_blocks, ignore_index=True) if dd_blocks else pd.DataFrame(
    columns=["stage","table","column","dtype","n_rows","n_nonnull","n_unique","null_rate","unique_ratio","sample_values"]
)

write_excel_no_sci(dd_all_raw, AUDIT_XLSX, "data_dictionary")


In [16]:
# execution: build audit workbook (Stage A), then pause before writing clean CSVs (Stage B)

artifacts = build_clean_audit_workbook(
    dfs_raw,
    audit_path=AUDIT_XLSX,
    ingestion_exceptions=ingestion_exceptions,
)

print(f"Audit workbook written: {artifacts['audit_path']}")
print("Review the clean_audit tab (blocks on top, column_map below) and the data_dictionary tab.")
print("Then run the Stage B gate cell to write cleaned CSV outputs.")


Audit workbook written: _clean_data\_clean_audit.xlsx
Review the clean_audit tab (blocks on top, column_map below) and the data_dictionary tab.
Then run the Stage B gate cell to write cleaned CSV outputs.


In [17]:
# human
# Stage B1a: apply automated clean_plan actions (baseline), write auto_audit, then STOP for allowlists
# note: Do NOT attempt to edit other cells while this one is paused at input(); edits won't affect this execution.

resp = input("Apply automated clean_plan now and write baseline auto_audit? Type YES to proceed: ").strip().upper()
if resp != "YES":
    print("Skipped applying automated clean_plan.")
    raise SystemExit("Execution halted by user.")

# Baseline clean (no delimiter splits, no dedup here)
dfs_clean_b, col_map_b, collisions_b, dropped_b, split_audit_df = apply_clean_plan(
    dfs_raw,
    split_allowlist=None,
)

# Keep artifacts consistent for later steps
artifacts["dfs_clean"] = dfs_clean_b
artifacts["col_map_all"] = col_map_b
artifacts["collisions_all"] = collisions_b
artifacts["dropped_cols_all"] = dropped_b

# Baseline auto_audit (shows schema hygiene changes only)
auto_summary, auto_details = build_auto_audit(
    dfs_raw=dfs_raw,
    dfs_clean=dfs_clean_b,
    col_map_all=col_map_b,
    collisions_all=collisions_b,
    dropped_cols_all=dropped_b,
)

_art = build_clean_audit_workbook(
    dfs_raw,
    audit_path=AUDIT_XLSX,
    ingestion_exceptions=ingestion_exceptions,
)

write_auto_audit_sheet(
    audit_path=artifacts["audit_path"],
    auto_summary=auto_summary,
    auto_details=auto_details,
    ingestion_exceptions=ingestion_exceptions,
    split_audit_df=pd.DataFrame(columns=["table","column","delimiter","max_splits","keep_original","strip_tokens","new_columns_added"]),
    sheet_name="auto_audit",
)

artifacts["audit_path"] = _art["audit_path"]


print(f"Baseline auto_audit written to: {artifacts['audit_path']}")
print("Now review _clean_audit.xlsx (clean_audit tab: delimited_candidates/samples + any dedup indicators).")
print("Then populate split_allowlist and dedup allowlists in the next cell and run it.")


Apply automated clean_plan now and write baseline auto_audit? Type YES to proceed:  yes


Baseline auto_audit written to: _clean_data\_clean_audit.xlsx
Now review _clean_audit.xlsx (clean_audit tab: delimited_candidates/samples + any dedup indicators).
Then populate split_allowlist and dedup allowlists in the next cell and run it.


In [18]:
# Stage B1b: build dedup detection audit (no data changes)
# Purpose: populates dedup_audit and dedup_plan sheets in _clean_audit.xlsx
# Review outputs before populating allowlists in cell below.

dedup_artifacts = build_dedup_audit_workbook(
    artifacts["dfs_clean"],
    audit_path=artifacts["audit_path"],
    max_candidate_keys_per_table=25,
)

print("dedup_audit and dedup_plan sheets written.")
print()

# Quick console summary so you don't have to open the file to assess
exact_summary = dedup_artifacts["exact_row_dup_summary_all"]
print("=== Exact Row Duplicate Summary ===")
print(exact_summary[["table","n_rows","dup_rows","dup_groups","dup_row_rate"]].to_string(index=False))
print()
print("Review dedup_audit tab in _clean_audit.xlsx for key offenders and samples.")
print("Then populate exact_row_tables / key_rules in the allowlists cell below.")


dedup_audit and dedup_plan sheets written.

=== Exact Row Duplicate Summary ===
                                table  n_rows  dup_rows  dup_groups  dup_row_rate
      GSCM 521 - Banner Grade Data v2     880         0           0      0.000000
     GSCM 521 - Canvas Gradebook Data     793       384         159      0.484237
GSCM 521 - Canvas Interaction Data v2  324148     63776       31888      0.196750

Review dedup_audit tab in _clean_audit.xlsx for key offenders and samples.
Then populate exact_row_tables / key_rules in the allowlists cell below.


# post cleaning cleaning/not automated

In [19]:
# Provide allowlists and apply post-clean transforms (delimiter splits + dedup), then pause for CSV export

# Delimiter handling
split_allowlist = [
    # {
    #     "table": "dim_customer",
    #     "column": "some_col",
    #     "delimiter": ";",
    #     "max_splits": 4,
    #     "keep_original": True,
    #     "strip_tokens": True,
    # }
]

# Apply delimiter splits (if any)
if split_allowlist:
    dfs_after_split, split_audit_df = apply_delimited_splits(
        artifacts["dfs_clean"],
        split_allowlist,
    )
    artifacts["dfs_clean"] = dfs_after_split
else:
    split_audit_df = pd.DataFrame(
        columns=["table","column","delimiter","max_splits","keep_original","strip_tokens","new_columns_added"]
    )

# Dedup allowlists
exact_row_tables = [
    #"GSCM 521 - Canvas Interaction Data v2",
]
key_rules = [
    # {"table": "dim_customer", "key_cols": ["customer_ref_id"], "keep": "max_nonnull"},
    # {"table": "dim_store", "key_cols": ["store_id"], "keep": "first"},
]

# Apply dedup (if any)
if exact_row_tables or key_rules:
    dfs_after_dedup, dedup_apply_audit = apply_dedup_allowlists(
        artifacts["dfs_clean"],
        exact_row_tables=exact_row_tables,
        key_rules=key_rules,
    )
    artifacts["dfs_clean"] = dfs_after_dedup
else:
    dedup_apply_audit = pd.DataFrame(
        columns=["table","mode","status","rows_before","rows_after","rows_removed","detail"]
    )

# Update auto_audit proof after post-clean transforms
auto_summary, auto_details = build_auto_audit(
    dfs_raw=dfs_raw,
    dfs_clean=artifacts["dfs_clean"],
    col_map_all=artifacts["col_map_all"],
    collisions_all=artifacts["collisions_all"],
    dropped_cols_all=artifacts["dropped_cols_all"],
)

write_auto_audit_sheet(
    audit_path=artifacts["audit_path"],
    auto_summary=auto_summary,
    auto_details=auto_details,
    ingestion_exceptions=ingestion_exceptions,
    split_audit_df=split_audit_df,
    sheet_name="auto_audit",
)

# Write dedup apply audit as its own sheet (clear, avoids expanding auto_audit further)
write_excel_no_sci(dedup_apply_audit, artifacts["audit_path"], "dedup_apply_audit")

print("Post-clean transforms applied.")
print("Review auto_audit (and dedup_apply_audit if present), then confirm CSV export.")


Post-clean transforms applied.
Review auto_audit (and dedup_apply_audit if present), then confirm CSV export.


In [ ]:
# dataset-specific manual cleaning — edit this section per project
#
# This is where you add any transformations that could not be detected automatically:
#   - Row filtering (e.g. remove rows matching a pattern in a specific column)
#   - Column renames (e.g. rename 'raw_col_name' -> 'standardized_name')
#   - Value replacements (e.g. fix known bad values in a categorical column)
#   - Explicit dedup on specific key columns (if not handled via dedup allowlists above)
#
# Pattern:
#   t = "table_name"
#   df = artifacts["dfs_clean"][t]
#   # ... your transforms ...
#   artifacts["dfs_clean"][t] = df
#
# Examples (commented out):
#
# # Row filtering
# mask = df["content_name"].astype("string").str.lower().str.contains(r"\.(?:gif|jpg|png)$", regex=True, na=False)
# artifacts["dfs_clean"][t] = df.loc[~mask].copy()
# print(f"[{t}] Removed {mask.sum()} rows matching image extension pattern")
#
# # Column rename
# if "old_col_name" in df.columns:
#     df = df.rename(columns={"old_col_name": "new_col_name"})
#
# # Value replacement
# df["some_col"] = df["some_col"].replace({"bad_value": "corrected_value"})
#
# # Explicit key-based dedup
# rows_before = len(df)
# df = df.drop_duplicates(subset=["key_col"])
# print(f"[{t}] Dedup: removed {rows_before - len(df)} rows")


In [22]:
# human

# Stage B2: final export gate
resp2 = input("Proceed to write cleaned CSV automated outputs now? Type YES to proceed: ").strip().upper()
if resp2 == "YES":
    write_clean_csv_outputs(artifacts["dfs_clean"], OUT_DIR, suffix="_clean")
    print(f"Clean CSVs written to: {OUT_DIR}")
else:
    print("Skipped writing cleaned CSVs.")
    raise SystemExit("Execution halted by user.")


Proceed to write cleaned CSV automated outputs now? Type YES to proceed:  yes


Clean CSVs written to: _clean_data
